# National Instability Classifier — Inference
## Per-Observation SHAP Artifacts on Unseen Data

Loads a trained XGBoost model from MLflow, runs predictions on new country-year
observations, and produces a self-contained SHAP artifact directory for every
observation — answering *"why did the model predict X for this country?"*

| Artifact per observation | File | Description |
|---|---|---|
| Waterfall plot | `waterfall.png` | Cumulative feature contributions from base value to final prediction |
| Force plot | `force.png` | Horizontal push/pull — red features drive collapse risk, blue drive stability |
| Feature contributions | `feature_contributions.csv` | Every feature with its raw value and SHAP value, sorted by \|SHAP\| |
| Prediction summary | `prediction_summary.json` | Probability, label, top risk drivers, top stabilizers |

### Prerequisites

1. A completed training run from `xgboost_instability_classifier.ipynb` — you need its MLflow **Run ID**.
2. New data in the **same raw schema** as training data (country_code, year, feature columns).
   Include enough historical years per country for lag/rolling features to be computed
   (at least `max(lag_years) + max(rolling_windows)` years of history before the
   first year you want a prediction for).

> **Sign convention:** positive SHAP → pushes toward collapse (class 1);
> negative SHAP → pushes toward stability (class 0).

In [ ]:
%%capture
%pip install \
    'azure-ai-ml>=1.12.0' \
    azure-identity \
    azureml-mlflow \
    azureml-fsspec \
    'xgboost>=2.0' \
    'scikit-learn>=1.3' \
    'shap>=0.44' \
    'matplotlib>=3.7' \
    'pandas>=2.0' \
    numpy \
    adlfs \
    fsspec \
    mlflow --quiet

In [ ]:
import json
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import shap
import mlflow
import mlflow.xgboost
from sklearn.preprocessing import LabelEncoder

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, ManagedIdentityCredential

warnings.filterwarnings('ignore', category=UserWarning)
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
INFER_CFG = {
    # ── Trained model ────────────────────────────────────────────────────────
    # Paste the Run ID from xgboost_instability_classifier.ipynb output.
    # Alternatively use a registered model URI: 'models:/instability-classifier/1'
    'run_id':       '<YOUR_TRAINING_RUN_ID>',
    'model_name':   'model',   # artifact path used when logging (default: 'model')

    # ── Azure ML workspace ───────────────────────────────────────────────────
    'subscription_id':  '<YOUR_SUBSCRIPTION_ID>',
    'resource_group':   '<YOUR_RESOURCE_GROUP>',
    'workspace_name':   '<YOUR_WORKSPACE_NAME>',

    # ── New data ─────────────────────────────────────────────────────────────
    # Same raw schema as training data.  Include historical years so that
    # lag and rolling features can be computed for the inference years.
    'use_datastore':      False,
    'datastore_name':     'adls_instability_data',
    'data_path':          'raw/country_year_panel_new.csv',
    'adls_account_name':  '<YOUR_STORAGE_ACCOUNT>',
    'adls_container':     '<YOUR_CONTAINER>',
    'adls_file_path':     'raw/country_year_panel_new.csv',
    'file_format':        'csv',

    # ── Schema (must match training exactly) ─────────────────────────────────
    'country_col':    'country_code',
    'year_col':       'year',
    'drop_cols':      ['country_name'],
    # Set target_col to None if the new data has no outcome column
    'target_col':     None,

    # ── Feature engineering (must match training exactly) ────────────────────
    'lag_features':     True,
    'lag_years':        [1, 2, 3],
    'rolling_windows':  [3, 5],
    'numeric_fill':     'median',

    # ── Inference scope ──────────────────────────────────────────────────────
    # Years to generate predictions and artifacts for.  Historical years
    # in the data are used only for feature engineering.
    'inference_years':  [2023, 2024],

    # ── SHAP ─────────────────────────────────────────────────────────────────
    'shap_top_n':       15,

    # ── Output ───────────────────────────────────────────────────────────────
    'output_dir':  'inference_outputs',
}

OUT_DIR = Path(INFER_CFG['output_dir'])
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory : {OUT_DIR.resolve()}")

In [ ]:
# Connect to Azure ML workspace and point MLflow at the workspace tracking URI
try:
    credential = ManagedIdentityCredential()
    ml_client = MLClient(
        credential,
        INFER_CFG['subscription_id'],
        INFER_CFG['resource_group'],
        INFER_CFG['workspace_name'],
    )
    ws = ml_client.workspaces.get(INFER_CFG['workspace_name'])
    logger.info('Connected via ManagedIdentityCredential')
except Exception:
    credential = DefaultAzureCredential()
    ml_client = MLClient(
        credential,
        INFER_CFG['subscription_id'],
        INFER_CFG['resource_group'],
        INFER_CFG['workspace_name'],
    )
    ws = ml_client.workspaces.get(INFER_CFG['workspace_name'])
    logger.info('Connected via DefaultAzureCredential')

mlflow.set_tracking_uri(ws.mlflow_tracking_uri)
logger.info('MLflow tracking URI: %s', ws.mlflow_tracking_uri)

In [ ]:
# Load the trained model from MLflow
model_uri = f"runs:/{INFER_CFG['run_id']}/{INFER_CFG['model_name']}"
model = mlflow.xgboost.load_model(model_uri)
logger.info('Loaded model from %s', model_uri)

# Retrieve the feature list from the logged model signature
run = mlflow.get_run(INFER_CFG['run_id'])
client = mlflow.tracking.MlflowClient()
model_info = client.get_run(INFER_CFG['run_id'])

# Load signature to get ordered feature names
from mlflow.models import Model
model_meta = Model.load(mlflow.artifacts.download_artifacts(
    run_id=INFER_CFG['run_id'],
    artifact_path=f"{INFER_CFG['model_name']}/MLmodel",
))
FEATURE_NAMES = [inp.name for inp in model_meta.signature.inputs.inputs]
logger.info('Feature count: %d', len(FEATURE_NAMES))
print(f'Feature count : {len(FEATURE_NAMES)}')
print(f'First 5       : {FEATURE_NAMES[:5]}')

In [ ]:
# Load new data from ADLS (same options as training notebook)
def _load_data(cfg, credential=None):
    fmt = cfg['file_format']

    def _read(f):
        if fmt == 'csv':
            return pd.read_csv(f)
        elif fmt == 'json':
            return pd.read_json(f, orient='records', lines=True)
        raise ValueError(f'Unsupported file_format: {fmt}')

    if cfg['use_datastore']:
        from azureml.fsspec import AzureMachineLearningFileSystem
        sub = ml_client._subscription_id
        rg  = ml_client._resource_group_name
        wsn = ml_client.workspace_name
        base = (f'azureml://subscriptions/{sub}/resourcegroups/{rg}'
                f'/workspaces/{wsn}/datastores/{cfg["datastore_name"]}/paths/')
        fs = AzureMachineLearningFileSystem(base)
        with fs.open(cfg['data_path'], 'rb') as fh:
            return _read(fh)
    else:
        import adlfs, os
        cred = credential or os.environ.get('ADLS_SAS_TOKEN') or os.environ.get('ADLS_ACCOUNT_KEY')
        fs = adlfs.AzureBlobFileSystem(account_name=cfg['adls_account_name'], credential=cred)
        with fs.open(f"{cfg['adls_container']}/{cfg['adls_file_path']}", 'rb') as fh:
            return _read(fh)


df_raw = _load_data(INFER_CFG)
logger.info('Raw data shape: %s', df_raw.shape)
df_raw.head()

In [ ]:
# Feature engineering — identical logic to the training notebook.
# The new data must include enough historical years per country for lags to be
# computed (at least max(lag_years) years before the first inference year).

def _years_since_last(grp, year_col, target_col):
    grp = grp.sort_values(year_col)
    last_event_year = None
    values = []
    for _, row in grp.iterrows():
        if last_event_year is None:
            values.append(float('nan'))
        else:
            values.append(row[year_col] - last_event_year)
        if target_col and row.get(target_col) == 1:
            last_event_year = row[year_col]
    return pd.Series(values, index=grp.index)


def engineer_features(df, cfg):
    country_col = cfg['country_col']
    year_col    = cfg['year_col']
    target_col  = cfg.get('target_col')   # may be None for unseen data

    df = df.copy().sort_values([country_col, year_col])

    cols_to_drop = [c for c in cfg.get('drop_cols', []) if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)

    exclude = {country_col, year_col}
    if target_col:
        exclude.add(target_col)
    feature_cols = [c for c in df.select_dtypes(include='number').columns if c not in exclude]

    if cfg.get('lag_features', True):
        for col in feature_cols:
            for lag in cfg.get('lag_years', [1, 2, 3]):
                df[f'{col}_lag{lag}'] = df.groupby(country_col)[col].shift(lag)
        for col in feature_cols:
            for window in cfg.get('rolling_windows', [3, 5]):
                df[f'{col}_roll{window}mean'] = (
                    df.groupby(country_col)[col]
                    .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
                )

    if target_col and target_col in df.columns:
        df['hist_instability_count'] = (
            df.groupby(country_col)[target_col]
            .transform(lambda s: s.shift(1).expanding().sum())
            .fillna(0)
        )
        df['years_since_last_event'] = (
            df.groupby(country_col, group_keys=False)
            .apply(lambda grp: _years_since_last(grp, year_col, target_col))
        )
    else:
        # Derive event history features from the data even without a target column
        # by using a proxy of zeros (no known events); update manually if history is known.
        df['hist_instability_count'] = 0.0
        df['years_since_last_event'] = float('nan')

    numeric_cols = [c for c in df.select_dtypes(include='number').columns
                    if c != target_col]
    fill = cfg.get('numeric_fill', 'median')
    if fill == 'median':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    elif fill == 'mean':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    else:
        df[numeric_cols] = df[numeric_cols].fillna(0)

    le = LabelEncoder()
    for col in df.select_dtypes(include=['object', 'category']).columns:
        if col != country_col:
            df[col] = le.fit_transform(df[col].astype(str))

    return df


df_engineered = engineer_features(df_raw, INFER_CFG)
logger.info('Engineered shape: %s', df_engineered.shape)

In [ ]:
# Filter to inference years and align columns to the model's feature list
country_col = INFER_CFG['country_col']
year_col    = INFER_CFG['year_col']

infer_mask = df_engineered[year_col].isin(INFER_CFG['inference_years'])
df_infer   = df_engineered[infer_mask].reset_index(drop=True)

# Metadata for labelling artifacts
df_infer_meta = df_infer[[country_col, year_col]].copy()

# Align feature columns to exactly what the model was trained on
missing_cols = [f for f in FEATURE_NAMES if f not in df_infer.columns]
extra_cols   = [f for f in df_infer.columns if f not in FEATURE_NAMES
                and f not in (country_col, year_col, INFER_CFG.get('target_col') or '')]

if missing_cols:
    logger.warning('Filling %d missing feature(s) with 0: %s', len(missing_cols), missing_cols)
    for col in missing_cols:
        df_infer[col] = 0.0
if extra_cols:
    logger.info('Dropping %d extra column(s) not seen at training: %s', len(extra_cols), extra_cols)

X_infer = df_infer[FEATURE_NAMES]

print(f'Inference observations : {len(X_infer)}')
print(f'Countries              : {df_infer_meta[country_col].nunique()}')
print(f'Years                  : {sorted(df_infer_meta[year_col].unique().tolist())}')

In [ ]:
# Run predictions
pred_proba = model.predict_proba(X_infer)[:, 1]
pred_label = (pred_proba >= 0.5).astype(int)

# Build SHAP explainer once — reused for every observation
explainer = shap.TreeExplainer(model)

# Quick summary
results_df = df_infer_meta.copy()
results_df['probability']     = pred_proba
results_df['predicted_label'] = pred_label
results_df = results_df.sort_values('probability', ascending=False)

print(f'Predictions generated for {len(results_df)} observations.')
print(f'Predicted collapse (label=1): {pred_label.sum()}')
results_df.head(10)

In [ ]:
def generate_observation_artifacts(
    row_idx,
    *,
    top_n=None,
    log_to_mlflow=False,
    mlflow_run_id=None,
):
    """Generate per-observation SHAP artifacts for a single inference row.

    Parameters
    ----------
    row_idx : int
        Positional index into X_infer / df_infer_meta.
    top_n : int, optional
        Max features to display in plots.  Defaults to INFER_CFG['shap_top_n'].
    log_to_mlflow : bool
        Log artifacts to an active MLflow run (useful when running inference
        as a tracked experiment; off by default for ad-hoc use).
    mlflow_run_id : str, optional
        Existing run to log into.  If None and log_to_mlflow=True, logs to
        whatever run is currently active.

    Returns
    -------
    dict — prediction summary (mirrored in prediction_summary.json)
    """
    top_n = top_n or INFER_CFG['shap_top_n']

    obs_country = df_infer_meta.iloc[row_idx][country_col]
    obs_year    = int(df_infer_meta.iloc[row_idx][year_col])
    obs_prob    = float(pred_proba[row_idx])
    obs_label   = int(pred_label[row_idx])

    # SHAP for this single observation
    X_obs    = X_infer.iloc[[row_idx]]
    shap_exp = explainer(X_obs)
    shap_vals = shap_exp.values[0]
    base_val  = float(shap_exp.base_values[0])
    feat_vals = X_obs.values[0]

    contrib_df = (
        pd.DataFrame({
            'feature':    FEATURE_NAMES,
            'value':      feat_vals,
            'shap_value': shap_vals,
        })
        .assign(abs_shap=lambda d: d['shap_value'].abs())
        .sort_values('abs_shap', ascending=False)
        .drop(columns='abs_shap')
        .reset_index(drop=True)
    )

    # Output directory for this observation
    safe_id = f"{obs_country}_{obs_year}"
    obs_dir = OUT_DIR / safe_id
    obs_dir.mkdir(parents=True, exist_ok=True)

    title = f"{obs_country} {obs_year}  |  P(humanitarian collapse) = {obs_prob:.3f}"

    # Waterfall plot
    plt.figure()
    shap.plots.waterfall(shap_exp[0], max_display=top_n, show=False)
    plt.title(title, pad=14)
    plt.tight_layout()
    wf_path = obs_dir / 'waterfall.png'
    plt.savefig(wf_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Force plot
    plt.figure()
    shap.plots.force(shap_exp[0], matplotlib=True, show=False)
    plt.title(title, pad=14)
    plt.tight_layout()
    force_path = obs_dir / 'force.png'
    plt.savefig(force_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Feature contribution CSV
    csv_path = obs_dir / 'feature_contributions.csv'
    contrib_df.to_csv(csv_path, index=False)

    # Prediction summary JSON
    def _rows(df_slice):
        return [
            {
                'feature': r['feature'],
                'value':   round(float(r['value']), 4),
                'shap':    round(float(r['shap_value']), 4),
            }
            for _, r in df_slice.iterrows()
        ]

    summary = {
        'country':          obs_country,
        'year':             obs_year,
        'probability':      round(obs_prob, 4),
        'predicted_label':  obs_label,
        'base_value':       round(base_val, 4),
        'top_risk_drivers': _rows(contrib_df[contrib_df['shap_value'] > 0].head(5)),
        'top_stabilizers':  _rows(contrib_df[contrib_df['shap_value'] < 0].head(5)),
    }

    json_path = obs_dir / 'prediction_summary.json'
    with open(json_path, 'w') as fh:
        json.dump(summary, fh, indent=2)

    if log_to_mlflow:
        mlflow_sub = f'inference/per_observation/{safe_id}'
        for path in (wf_path, force_path, csv_path, json_path):
            mlflow.log_artifact(str(path), artifact_path=mlflow_sub)

    logger.info(
        '%s %d : P=%.3f  top driver="%s" (SHAP=%+.3f)',
        obs_country, obs_year, obs_prob,
        contrib_df.iloc[0]['feature'], contrib_df.iloc[0]['shap_value'],
    )
    return summary

In [ ]:
# Generate per-observation artifacts for every inference observation
all_summaries = []

for i in range(len(X_infer)):
    try:
        summary = generate_observation_artifacts(i)
        all_summaries.append(summary)
    except Exception as exc:
        country = df_infer_meta.iloc[i][country_col]
        year    = df_infer_meta.iloc[i][year_col]
        logger.warning('Skipped %s %s: %s', country, year, exc)

# Write consolidated inference results
summary_csv_path = OUT_DIR / 'inference_results.csv'
summary_json_path = OUT_DIR / 'inference_results.json'

pd.DataFrame([
    {
        'country':         s['country'],
        'year':            s['year'],
        'probability':     s['probability'],
        'predicted_label': s['predicted_label'],
        'top_driver':      s['top_risk_drivers'][0]['feature'] if s['top_risk_drivers'] else '',
        'top_driver_shap': s['top_risk_drivers'][0]['shap']    if s['top_risk_drivers'] else 0.0,
    }
    for s in all_summaries
]).sort_values('probability', ascending=False).to_csv(summary_csv_path, index=False)

with open(summary_json_path, 'w') as fh:
    json.dump(all_summaries, fh, indent=2)

print(f'Done. {len(all_summaries)} observations processed.')
print(f'Per-observation artifacts : {OUT_DIR}/')
print(f'Consolidated results CSV  : {summary_csv_path}')

In [ ]:
# Print a ranked summary table
print(f"{'Country':<10} {'Year':>6} {'P(collapse)':>13} {'Label':>8}  Top risk driver")
print('─' * 80)
for s in sorted(all_summaries, key=lambda x: x['probability'], reverse=True):
    label_str  = 'COLLAPSE' if s['predicted_label'] else 'stable'
    top_driver = s['top_risk_drivers'][0]['feature'] if s['top_risk_drivers'] else '—'
    top_shap   = s['top_risk_drivers'][0]['shap']    if s['top_risk_drivers'] else 0.0
    print(
        f"{s['country']:<10} {s['year']:>6} {s['probability']:>13.4f} "
        f"{label_str:>8}  {top_driver} (SHAP={top_shap:+.3f})"
    )

## Output Structure

```
inference_outputs/
  inference_results.csv          ← all countries ranked by P(collapse)
  inference_results.json         ← same, with full top-driver detail
  BWA_2024/
    waterfall.png                ← primary explainability plot
    force.png                    ← horizontal push/pull diagram
    feature_contributions.csv    ← all features + SHAP values
    prediction_summary.json      ← structured summary
  SOM_2024/
    ...
```

### Reading the waterfall
- The plot starts at **E[f(x)]** — the model's average log-odds across training data.
- Each bar adds one feature's SHAP contribution (red = toward collapse, blue = toward stability).
- The final value is **f(x)**, which maps to the reported probability via the logistic function.
- Features are ordered top-to-bottom by absolute SHAP magnitude — the most influential for
  this specific country-year appear first.

### Updating for a new prediction cycle
1. Replace `adls_file_path` in `INFER_CFG` with your new data file.
2. Update `inference_years` to the target year(s).
3. Run all cells — artifacts are written to `inference_outputs/` and overwrite previous runs
   for the same country-year if you re-run with the same output directory.